# Return on Investment: Federal School Mental Health Workforce Pipeline

## A Two-Year Analysis of the MHSP and SBMH Grant Programs

---

### Background

The United States faces a persistent school mental health workforce shortage. The student-to-school-psychologist ratio nationally exceeds 1,000:1 — more than double the NASP-recommended 500:1. In response, two federal grant programs channel funding to build this pipeline from both ends:

- **MHSP (Mental Health Service Professionals)** funds higher education institutions to train and place school-based mental health providers — psychologists, counselors, and social workers.
- **SBMH (School-Based Mental Health)** funds local and state education agencies (LEAs/SEAs) to hire, retain, and expand access to school mental health services directly.

This analysis examines grantee performance across **Year 1 and Year 2** of the grant cycle, asking:

> *Did these grants deliver return on their investment — and which grantees, regions, and program features drove the most impact?*

### Data
All data are synthetic and anonymized. The schema mirrors real federal APR (Annual Performance Report) data structures. Analyses use GPRA (Government Performance and Results Act) measures as the primary outcome indicators.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import json

warnings.filterwarnings('ignore')

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.family': 'DejaVu Sans',
})
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#4CAF50', '#9E5EA8']

# ── Load data ─────────────────────────────────────────────────────────────────
DATA_DIR = os.path.join(os.getcwd(), 'data')

mhsp_y1 = pd.read_excel(os.path.join(DATA_DIR, 'APRs_Year1.xlsx'), sheet_name='MHSP')
sbmh_y1 = pd.read_excel(os.path.join(DATA_DIR, 'APRs_Year1.xlsx'), sheet_name='SBMH')
mhsp_y2 = pd.read_excel(os.path.join(DATA_DIR, 'MHSP_Year2.xlsx'))
sbmh_y2 = pd.read_excel(os.path.join(DATA_DIR, 'SBMH_Year2.xlsx'))

print(f"MHSP Year 1: {mhsp_y1.shape[0]} grantees")
print(f"SBMH Year 1: {sbmh_y1.shape[0]} grantees")
print(f"MHSP Year 2: {mhsp_y2.shape[0]} grantees")
print(f"SBMH Year 2: {sbmh_y2.shape[0]} grantees")

## 2. Data Preparation

### 2.1 Merge Year 1 and Year 2

Year 1 APR data (from `APRs_Year1.xlsx`) is joined to Year 2 GPRA actuals using grant ID as the merge key. The goal is a longitudinal view of each grantee's performance trajectory across both reporting periods.

In [ ]:
def clean_numeric(series, sentinel=999):
    """Coerce to numeric and replace sentinel values (999 = not reported) with NaN."""
    return pd.to_numeric(series, errors='coerce').replace(sentinel, np.nan)


# ── MHSP merge ────────────────────────────────────────────────────────────────
mhsp_y2_slim = mhsp_y2[[
    'grant_id',
    'gpra_1a_target_y2', 'gpra_1a_actual_y2',
    'gpra_1b_target_y2', 'gpra_1b_actual_y2',
    'gpra_2a_target_y2', 'gpra_2a_actual_y2',
    'gpra_2b_target_y2', 'gpra_2b_actual_y2',
    'gpra_3a_target_y2', 'gpra_3a_actual_y2',
    'gpra_3b_target_y2', 'gpra_3b_actual_y2',
]].copy()

mhsp = mhsp_y1.merge(
    mhsp_y2_slim,
    how='left',
    left_on='grantee_number',
    right_on='grant_id'
).drop(columns=['grant_id'])

# ── SBMH merge ────────────────────────────────────────────────────────────────
sbmh_y2_slim = sbmh_y2[[
    'grant_id',
    'gpra_1_target_y2', 'gpra_1_actual_y2',
    'gpra_2_target_y2', 'gpra_2_actual_y2',
    'gpra_4_target_y2', 'gpra_4_actual_y2',
    'gpra_5_target_y2', 'gpra_5_actual_y2',
]].copy()

sbmh = sbmh_y1.merge(
    sbmh_y2_slim,
    how='left',
    left_on='grantee_number',
    right_on='grant_id'
).drop(columns=['grant_id'])

# ── Clean key numeric columns ─────────────────────────────────────────────────
# MHSP
mhsp['trained_y1']     = clean_numeric(mhsp['gpra1_a_annual_raw_actual'])
mhsp['trained_y2']     = clean_numeric(mhsp['gpra_1a_actual_y2'])
mhsp['placed_y1']      = clean_numeric(mhsp['gpra1_b_current_raw_actual'])
mhsp['placed_y2']      = clean_numeric(mhsp['gpra_1b_actual_y2'])
mhsp['retained_y1']    = clean_numeric(mhsp['gpra2_a_annual_raw_actual'])
mhsp['retained_y2']    = clean_numeric(mhsp['gpra_2a_actual_y2'])
mhsp['planned_total']  = clean_numeric(mhsp['planned_number_trained'])

# SBMH
sbmh['hired_y1']       = clean_numeric(sbmh['gpra1_raw_actual'])
sbmh['hired_y2']       = clean_numeric(sbmh['gpra_1_actual_y2'])
sbmh['retained_y1']    = clean_numeric(sbmh['gpra2_raw_actual'])
sbmh['retained_y2']    = clean_numeric(sbmh['gpra_2_actual_y2'])
sbmh['students_y1']    = clean_numeric(sbmh['gpra5_raw_actual'])
sbmh['students_y2']    = clean_numeric(sbmh['gpra_5_actual_y2'])
sbmh['student_pop']    = clean_numeric(sbmh['student_population'])
sbmh['psych_ratio']    = clean_numeric(sbmh['students_per_psych'])
sbmh['counselor_ratio']= clean_numeric(sbmh['students_per_counselor'])

matched_mhsp = mhsp['trained_y2'].notna().sum()
matched_sbmh = sbmh['hired_y2'].notna().sum()
print(f"MHSP grantees with Year 2 data: {matched_mhsp} / {len(mhsp)}")
print(f"SBMH grantees with Year 2 data: {matched_sbmh} / {len(sbmh)}")

### 2.2 Program-Level Summary

Before diving into trends, we establish what the two programs produced in total across both years.

In [ ]:
summary = {
    'Program': ['MHSP', 'MHSP', 'SBMH', 'SBMH'],
    'Year': ['Year 1', 'Year 2', 'Year 1', 'Year 2'],
    'Providers Trained/Hired': [
        int(mhsp['trained_y1'].sum(skipna=True)),
        int(mhsp['trained_y2'].sum(skipna=True)),
        int(sbmh['hired_y1'].sum(skipna=True)),
        int(sbmh['hired_y2'].sum(skipna=True)),
    ],
    'Providers Placed/Retained': [
        int(mhsp['placed_y1'].sum(skipna=True)),
        int(mhsp['placed_y2'].sum(skipna=True)),
        int(sbmh['retained_y1'].sum(skipna=True)),
        int(sbmh['retained_y2'].sum(skipna=True)),
    ],
    'Students Reached (SBMH)': [
        '',
        '',
        f"{int(sbmh['students_y1'].sum(skipna=True)):,}",
        f"{int(sbmh['students_y2'].sum(skipna=True)):,}",
    ]
}
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

## 3. Year-Over-Year Performance Growth

The core ROI question: did grantee output grow from Year 1 to Year 2?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Year-Over-Year Growth: Key Workforce Pipeline Metrics', fontsize=14, fontweight='bold', y=1.01)

def yoy_bar(ax, y1_vals, y2_vals, title, ylabel, color1, color2):
    """Plot paired Y1/Y2 totals as grouped bars."""
    totals = [y1_vals.sum(skipna=True), y2_vals.sum(skipna=True)]
    bars = ax.bar(['Year 1', 'Year 2'], totals, color=[color1, color2], width=0.5, edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, totals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + totals[1]*0.02,
                f'{int(val):,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    pct = ((totals[1] - totals[0]) / totals[0] * 100) if totals[0] > 0 else 0
    ax.set_ylim(0, max(totals) * 1.18)
    ax.text(0.5, 0.93, f'Δ {pct:+.0f}%', transform=ax.transAxes,
            ha='center', fontsize=11, color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='none'))

yoy_bar(axes[0], mhsp['trained_y1'], mhsp['trained_y2'],
        'MHSP: Trainees\n(Providers Trained Annually)', 'Total Trainees',
        PALETTE[0], '#1a5276')

yoy_bar(axes[1], sbmh['hired_y1'], sbmh['hired_y2'],
        'SBMH: Providers Hired', 'Total Hired',
        PALETTE[2], '#b7770d')

yoy_bar(axes[2], sbmh['students_y1'], sbmh['students_y2'],
        'SBMH: Students Reached\nby Grant-Funded Providers', 'Students',
        PALETTE[3], '#2d6a4f')

plt.tight_layout()
plt.savefig('figures/fig1_yoy_growth.png', bbox_inches='tight')
plt.show()
print("Figure saved.")

## 4. MHSP Deep Dive: The Training-to-Placement Pipeline

MHSP grantees operate a multi-stage pipeline: they **train** students in mental health programs, then support **placement** in school settings, and finally track **retention** in those positions. Each stage is a potential drop-off point.

Here we examine conversion rates — what share of trainees ultimately reached students in schools.

In [ ]:
# Pipeline funnel: trained → placed → retained (Y2 values for most complete picture)
pipeline = {
    'Stage': ['Trained (Y1)', 'Trained (Y2)', 'Placed (Y1)', 'Placed (Y2)', 'Retained (Y1)', 'Retained (Y2)'],
    'Count': [
        mhsp['trained_y1'].sum(skipna=True),
        mhsp['trained_y2'].sum(skipna=True),
        mhsp['placed_y1'].sum(skipna=True),
        mhsp['placed_y2'].sum(skipna=True),
        mhsp['retained_y1'].sum(skipna=True),
        mhsp['retained_y2'].sum(skipna=True),
    ],
    'Year': ['Y1','Y2','Y1','Y2','Y1','Y2'],
    'Stage_Label': ['Trained','Trained','Placed','Placed','Retained','Retained']
}
pipeline_df = pd.DataFrame(pipeline)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MHSP Workforce Pipeline: Training → Placement → Retention', fontsize=13, fontweight='bold')

# Left: Grouped bar by stage and year
ax = axes[0]
x = np.arange(3)
width = 0.35
stages = ['Trained', 'Placed', 'Retained']
y1_vals = [pipeline_df[pipeline_df['Stage_Label']==s].set_index('Year').loc['Y1','Count'] for s in stages]
y2_vals = [pipeline_df[pipeline_df['Stage_Label']==s].set_index('Year').loc['Y2','Count'] for s in stages]

b1 = ax.bar(x - width/2, y1_vals, width, label='Year 1', color=PALETTE[0], alpha=0.85)
b2 = ax.bar(x + width/2, y2_vals, width, label='Year 2', color='#1a5276', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylabel('Total Providers')
ax.set_title('Pipeline Volume by Stage')
ax.legend()
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{int(h):,}',
                ha='center', va='bottom', fontsize=9)

# Right: Provider type breakdown Y1
ax2 = axes[1]
provider_counts = {
    'Counselors': mhsp['provider_type_counselors'].sum(skipna=True),
    'Psychologists': mhsp['provider_type_psychologists'].sum(skipna=True),
    'Social Workers': mhsp['provider_type_social_workers'].sum(skipna=True),
}
colors_pie = [PALETTE[0], PALETTE[1], PALETTE[2]]
wedges, texts, autotexts = ax2.pie(
    provider_counts.values(),
    labels=provider_counts.keys(),
    colors=colors_pie,
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax2.set_title('Provider Types Trained (MHSP)')

plt.tight_layout()
plt.savefig('figures/fig2_mhsp_pipeline.png', bbox_inches='tight')
plt.show()

## 5. SBMH Deep Dive: Student Access and Provider Ratios

SBMH grantees are evaluated on how many students gain access to school mental health services. A key indicator is the **student-to-provider ratio** — how many students each grant-funded provider serves.

The NASP-recommended ceiling is 500:1 for psychologists. We visualize where grantees fall relative to this benchmark.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SBMH: Student Access to School Mental Health Services', fontsize=13, fontweight='bold')

# Left: Students served Y1 vs Y2 by region
ax = axes[0]
region_students = sbmh.groupby('region').agg(
    y1=('students_y1', 'sum'),
    y2=('students_y2', 'sum')
).reset_index().dropna()
region_order = region_students.sort_values('y2', ascending=False)['region'].tolist()
x = np.arange(len(region_order))
width = 0.35
y1r = [region_students[region_students['region']==r]['y1'].values[0] for r in region_order]
y2r = [region_students[region_students['region']==r]['y2'].values[0] for r in region_order]
ax.bar(x - width/2, y1r, width, label='Year 1', color=PALETTE[3], alpha=0.8)
ax.bar(x + width/2, y2r, width, label='Year 2', color='#1b4332', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(region_order)
ax.set_ylabel('Total Students Reached')
ax.set_title('Students Reached by Region (Y1 vs Y2)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
ax.legend()

# Right: Student-to-psychologist ratio distribution vs 500:1 benchmark
ax2 = axes[1]
ratio_data = sbmh['psych_ratio'].dropna()
ratio_data = ratio_data[ratio_data > 0]  # exclude zeros
ax2.hist(ratio_data, bins=25, color=PALETTE[1], edgecolor='white', alpha=0.85)
ax2.axvline(500, color='#e74c3c', linewidth=2, linestyle='--', label='NASP Recommended (500:1)')
ax2.axvline(ratio_data.median(), color='#2c3e50', linewidth=2, linestyle='-',
            label=f'Median ({ratio_data.median():.0f}:1)')
ax2.set_xlabel('Students per School Psychologist')
ax2.set_ylabel('Number of Grantees')
ax2.set_title('Student-to-Psychologist Ratio\nvs. NASP Recommended Benchmark')
ax2.legend(fontsize=9)
pct_above = (ratio_data > 500).mean() * 100
ax2.text(0.97, 0.95, f'{pct_above:.0f}% above\nrecommended ratio',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdecea', edgecolor='#e74c3c'))

plt.tight_layout()
plt.savefig('figures/fig3_sbmh_access.png', bbox_inches='tight')
plt.show()

## 6. Regional Variation in Program Impact

Federal grants flow to all regions, but not all regions perform equally. This section examines whether certain regions produced disproportionate workforce gains — and whether geographic patterns suggest where future investment would have the greatest marginal impact.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Regional Variation in Grant Program Outcomes', fontsize=13, fontweight='bold')

# MHSP: trained per grantee by region
ax = axes[0]
mhsp_reg = mhsp.groupby('region').agg(
    trained_y1_mean=('trained_y1', 'mean'),
    trained_y2_mean=('trained_y2', 'mean'),
    n=('trained_y1', 'count')
).reset_index().dropna()
mhsp_reg = mhsp_reg.sort_values('trained_y2_mean', ascending=True)

y = np.arange(len(mhsp_reg))
ax.barh(y - 0.2, mhsp_reg['trained_y1_mean'], height=0.35, color=PALETTE[0], alpha=0.8, label='Year 1 Avg')
ax.barh(y + 0.2, mhsp_reg['trained_y2_mean'], height=0.35, color='#1a5276', alpha=0.8, label='Year 2 Avg')
ax.set_yticks(y)
ax.set_yticklabels(mhsp_reg['region'])
ax.set_xlabel('Avg Trainees per Grantee')
ax.set_title('MHSP: Average Trainees per\nGrantee by Region')
ax.legend(fontsize=9)

# SBMH: avg students served per grantee by region
ax2 = axes[1]
sbmh_reg = sbmh.groupby('region').agg(
    students_y1_mean=('students_y1', 'mean'),
    students_y2_mean=('students_y2', 'mean'),
).reset_index().dropna()
sbmh_reg = sbmh_reg.sort_values('students_y2_mean', ascending=True)

y2 = np.arange(len(sbmh_reg))
ax2.barh(y2 - 0.2, sbmh_reg['students_y1_mean'], height=0.35, color=PALETTE[3], alpha=0.8, label='Year 1 Avg')
ax2.barh(y2 + 0.2, sbmh_reg['students_y2_mean'], height=0.35, color='#1b4332', alpha=0.8, label='Year 2 Avg')
ax2.set_yticks(y2)
ax2.set_yticklabels(sbmh_reg['region'])
ax2.set_xlabel('Avg Students Reached per Grantee')
ax2.set_title('SBMH: Average Students Reached\nper Grantee by Region')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig4_regional_variation.png', bbox_inches='tight')
plt.show()

## 7. Diversity and Equity Features

MHSP grantees were required to indicate whether they incorporated specific diversity, equity, and inclusion (DEI) strategies into their training programs. This section examines which strategies were most commonly adopted and whether DEI-focused grantees showed different training outcomes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MHSP Diversity & Equity Features: Adoption and Outcomes', fontsize=13, fontweight='bold')

# Left: DEI feature adoption rates
ax = axes[0]
dei_cols = {
    'Race/Ethnicity Focus': 'diversity_race',
    'Language Access': 'diversity_language',
    'Color/Identity': 'diversity_color',
    'Disability': 'diversity_disability',
    'National Origin': 'diversity_national_origin',
    'Gender': 'diversity_gender',
    'Low Income': 'diversity_low_income',
    'Rural': 'diversity_rural',
    'Age': 'diversity_age',
    'First Generation': 'diversity_first_gen',
}

dei_rates = {}
for label, col in dei_cols.items():
    if col in mhsp.columns:
        vals = pd.to_numeric(mhsp[col], errors='coerce')
        dei_rates[label] = vals.mean() * 100

dei_df = pd.DataFrame(list(dei_rates.items()), columns=['Feature', 'Adoption %'])
dei_df = dei_df.sort_values('Adoption %', ascending=True)
colors = [PALETTE[0] if v >= 30 else '#aec6cf' for v in dei_df['Adoption %']]
ax.barh(dei_df['Feature'], dei_df['Adoption %'], color=colors, edgecolor='white')
ax.set_xlabel('% of MHSP Grantees Adopting')
ax.set_title('DEI Feature Adoption\nAcross MHSP Grantees')
ax.axvline(50, color='gray', linestyle=':', alpha=0.6)
ax.text(51, -0.5, '50%', fontsize=8, color='gray')

# Right: Grantees with vs without mentorship for providers of color — training outcomes
ax2 = axes[1]
mhsp['mentorship_poc_flag'] = pd.to_numeric(mhsp['mentorship_poc'], errors='coerce').fillna(0).astype(int)
mentorship_comp = mhsp.groupby('mentorship_poc_flag').agg(
    y1=('trained_y1', 'mean'),
    y2=('trained_y2', 'mean'),
    n=('trained_y1', 'count')
).reset_index()
mentorship_comp['label'] = mentorship_comp['mentorship_poc_flag'].map({0: 'No Mentorship\nProgram', 1: 'Mentorship for\nProviders of Color'})

x = np.arange(len(mentorship_comp))
ax2.bar(x - 0.2, mentorship_comp['y1'], width=0.35, color=PALETTE[0], alpha=0.8, label='Year 1')
ax2.bar(x + 0.2, mentorship_comp['y2'], width=0.35, color='#1a5276', alpha=0.8, label='Year 2')
ax2.set_xticks(x)
ax2.set_xticklabels(mentorship_comp['label'])
ax2.set_ylabel('Avg Trainees per Grantee')
ax2.set_title('Training Output: Grantees With vs.\nWithout Mentorship for Providers of Color')
ax2.legend()
for i, row in mentorship_comp.iterrows():
    ax2.text(i, max(row['y1'], row['y2']) + 0.3, f'n={int(row["n"])}', ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('figures/fig5_dei_features.png', bbox_inches='tight')
plt.show()

## 8. Grantee-Level Performance Scatterplot

We now visualize individual grantee trajectories to understand variability. For each grantee with both Year 1 and Year 2 data, we plot Year 1 output against Year 2 output. Points above the diagonal represent **growth**; points below represent **decline**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Grantee-Level Trajectories: Year 1 vs. Year 2 Performance', fontsize=13, fontweight='bold')

def scatter_yoy(ax, df, y1_col, y2_col, title, xlabel, ylabel, color, cap=None):
    """Scatter plot of Y1 vs Y2 with diagonal reference line."""
    sub = df[[y1_col, y2_col]].dropna()
    if cap:
        sub = sub[(sub[y1_col] <= cap) & (sub[y2_col] <= cap)]
    x_vals, y_vals = sub[y1_col], sub[y2_col]
    improved = (y_vals > x_vals).sum()
    declined = (y_vals < x_vals).sum()
    same = (y_vals == x_vals).sum()
    ax.scatter(x_vals, y_vals, color=color, alpha=0.55, s=40, edgecolors='white', linewidth=0.5)
    lim = max(x_vals.max(), y_vals.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, label='No change')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.text(0.03, 0.97, f'↑ Improved: {improved}\n→ Same: {same}\n↓ Declined: {declined}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#cccccc'))

scatter_yoy(axes[0], mhsp, 'trained_y1', 'trained_y2',
            'MHSP: Trainees\n(Year 1 vs. Year 2)',
            'Year 1 Trainees', 'Year 2 Trainees', PALETTE[0], cap=100)

scatter_yoy(axes[1], sbmh, 'students_y1', 'students_y2',
            'SBMH: Students Reached\n(Year 1 vs. Year 2)',
            'Year 1 Students', 'Year 2 Students', PALETTE[3], cap=30000)

plt.tight_layout()
plt.savefig('figures/fig6_grantee_trajectories.png', bbox_inches='tight')
plt.show()
print("Note: Outliers above the cap threshold excluded for readability.")

## 9. SBMH Hire Type Mix and Student Reach

SBMH grantees hired different types of providers (psychologists, social workers, counselors). Does the provider mix predict how many students a grantee reaches?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SBMH Provider Hire Type: Mix and Student Reach', fontsize=13, fontweight='bold')

# Left: Hire type mix
ax = axes[0]
hire_counts = {
    'Counselors': sbmh['hire_type_counselors'].sum(skipna=True),
    'Social Workers': sbmh['hire_type_ssw'].sum(skipna=True),
    'Psychologists': sbmh['hire_type_psychologists'].sum(skipna=True),
}
wedges, texts, autotexts = ax.pie(
    hire_counts.values(), labels=hire_counts.keys(),
    colors=[PALETTE[0], PALETTE[2], PALETTE[1]],
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax.set_title('SBMH Grantees: Provider\nType Hired')

# Right: Students reached by LEA vs SEA grantees
ax2 = axes[1]
lea_sea = sbmh.groupby('lea_sea').agg(
    y1=('students_y1', 'median'),
    y2=('students_y2', 'median'),
    n=('students_y1', 'count')
).reset_index().dropna()

x = np.arange(len(lea_sea))
ax2.bar(x - 0.2, lea_sea['y1'], width=0.35, color=PALETTE[3], alpha=0.8, label='Year 1 Median')
ax2.bar(x + 0.2, lea_sea['y2'], width=0.35, color='#1b4332', alpha=0.8, label='Year 2 Median')
ax2.set_xticks(x)
ax2.set_xticklabels([f"{row['lea_sea']}\n(n={int(row['n'])})" for _, row in lea_sea.iterrows()])
ax2.set_ylabel('Median Students Reached')
ax2.set_title('Median Students Reached:\nLEA vs. SEA Grantees')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.1f}K'))
ax2.legend()

plt.tight_layout()
plt.savefig('figures/fig7_hire_type_reach.png', bbox_inches='tight')
plt.show()

## 10. Retention as a Signal of Program Sustainability

Hiring a provider is one thing — keeping them in the school is another. Retention rates offer a window into program sustainability. We examine how SBMH grantees' Year 1 retention compares to Year 2, and whether high-retention programs reach more students.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Retention: Sustainability of the SBMH Workforce', fontsize=13, fontweight='bold')

# Left: Retention distribution Y1 vs Y2
ax = axes[0]
r1 = sbmh['retained_y1'].dropna()
r2 = sbmh['retained_y2'].dropna()
r1 = r1[r1 > 0]
r2 = r2[r2 > 0]
ax.hist(r1, bins=20, color=PALETTE[3], alpha=0.6, label=f'Year 1 (n={len(r1)})', edgecolor='white')
ax.hist(r2, bins=20, color='#1b4332', alpha=0.6, label=f'Year 2 (n={len(r2)})', edgecolor='white')
ax.axvline(r1.median(), color=PALETTE[3], linestyle='--', linewidth=2)
ax.axvline(r2.median(), color='#1b4332', linestyle='--', linewidth=2)
ax.set_xlabel('Providers Retained')
ax.set_ylabel('Number of Grantees')
ax.set_title('Distribution of Provider Retention\nYear 1 vs Year 2')
ax.legend()
ax.set_xlim(0, 200)

# Right: Retention vs. students reached (Y2)
ax2 = axes[1]
corr_df = sbmh[['retained_y2','students_y2']].dropna()
corr_df = corr_df[(corr_df['retained_y2'] > 0) & (corr_df['students_y2'] > 0)]
corr_df = corr_df[(corr_df['retained_y2'] < 200) & (corr_df['students_y2'] < 60000)]
ax2.scatter(corr_df['retained_y2'], corr_df['students_y2'],
            color=PALETTE[1], alpha=0.6, s=40, edgecolors='white')
# Trend line
z = np.polyfit(corr_df['retained_y2'], corr_df['students_y2'], 1)
p = np.poly1d(z)
x_line = np.linspace(corr_df['retained_y2'].min(), corr_df['retained_y2'].max(), 100)
ax2.plot(x_line, p(x_line), color='#2c3e50', linewidth=2, linestyle='--', alpha=0.8)
corr_val = corr_df.corr()['retained_y2']['students_y2']
ax2.text(0.97, 0.05, f'r = {corr_val:.2f}', transform=ax2.transAxes,
         ha='right', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#cccccc'))
ax2.set_xlabel('Providers Retained (Year 2)')
ax2.set_ylabel('Students Reached (Year 2)')
ax2.set_title('Retention vs. Student Reach\n(SBMH Year 2)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))

plt.tight_layout()
plt.savefig('figures/fig8_retention.png', bbox_inches='tight')
plt.show()

## 11. Summary Statistics Export

In [ ]:
os.makedirs('output', exist_ok=True)

# ── Program-level summary ─────────────────────────────────────────────────────
program_summary = pd.DataFrame({
    'program': ['MHSP', 'MHSP', 'SBMH', 'SBMH'],
    'year': ['Year 1', 'Year 2', 'Year 1', 'Year 2'],
    'n_grantees': [
        mhsp['trained_y1'].notna().sum(), mhsp['trained_y2'].notna().sum(),
        sbmh['hired_y1'].notna().sum(), sbmh['hired_y2'].notna().sum(),
    ],
    'total_trained_hired': [
        int(mhsp['trained_y1'].sum(skipna=True)), int(mhsp['trained_y2'].sum(skipna=True)),
        int(sbmh['hired_y1'].sum(skipna=True)), int(sbmh['hired_y2'].sum(skipna=True)),
    ],
    'total_placed_retained': [
        int(mhsp['placed_y1'].sum(skipna=True)), int(mhsp['placed_y2'].sum(skipna=True)),
        int(sbmh['retained_y1'].sum(skipna=True)), int(sbmh['retained_y2'].sum(skipna=True)),
    ],
    'total_students_reached': [
        None, None,
        int(sbmh['students_y1'].sum(skipna=True)), int(sbmh['students_y2'].sum(skipna=True)),
    ]
})

# ── Regional breakdown ────────────────────────────────────────────────────────
mhsp_region_summary = mhsp.groupby('region').agg(
    n_grantees=('trained_y1', 'count'),
    total_trained_y1=('trained_y1', 'sum'),
    total_trained_y2=('trained_y2', 'sum'),
    avg_trained_y1=('trained_y1', 'mean'),
    avg_trained_y2=('trained_y2', 'mean'),
).round(1).reset_index()
mhsp_region_summary.insert(0, 'program', 'MHSP')

sbmh_region_summary = sbmh.groupby('region').agg(
    n_grantees=('hired_y1', 'count'),
    total_hired_y1=('hired_y1', 'sum'),
    total_hired_y2=('hired_y2', 'sum'),
    total_students_y1=('students_y1', 'sum'),
    total_students_y2=('students_y2', 'sum'),
    avg_psych_ratio=('psych_ratio', 'mean'),
).round(1).reset_index()
sbmh_region_summary.insert(0, 'program', 'SBMH')

# Save
program_summary.to_csv('output/program_summary.csv', index=False)
mhsp_region_summary.to_csv('output/mhsp_region_summary.csv', index=False)
sbmh_region_summary.to_csv('output/sbmh_region_summary.csv', index=False)

# JSON for quick reference
roi_stats = {
    'mhsp_total_trained_y1': int(mhsp['trained_y1'].sum(skipna=True)),
    'mhsp_total_trained_y2': int(mhsp['trained_y2'].sum(skipna=True)),
    'mhsp_yoy_change_pct': round((mhsp['trained_y2'].sum(skipna=True) - mhsp['trained_y1'].sum(skipna=True))
                                  / max(mhsp['trained_y1'].sum(skipna=True), 1) * 100, 1),
    'sbmh_total_hired_y1': int(sbmh['hired_y1'].sum(skipna=True)),
    'sbmh_total_hired_y2': int(sbmh['hired_y2'].sum(skipna=True)),
    'sbmh_total_students_y1': int(sbmh['students_y1'].sum(skipna=True)),
    'sbmh_total_students_y2': int(sbmh['students_y2'].sum(skipna=True)),
    'sbmh_students_yoy_change_pct': round((sbmh['students_y2'].sum(skipna=True) - sbmh['students_y1'].sum(skipna=True))
                                          / max(sbmh['students_y1'].sum(skipna=True), 1) * 100, 1),
    'sbmh_pct_above_nasp_ratio': round((sbmh['psych_ratio'].dropna() > 500).mean() * 100, 1),
    'sbmh_median_psych_ratio': round(sbmh['psych_ratio'].dropna().median(), 0),
}
with open('output/roi_summary_stats.json', 'w') as f:
    json.dump(roi_stats, f, indent=2)

print("Outputs saved to output/ directory.")
print(json.dumps(roi_stats, indent=2))

## 12. Key Findings

---

### MHSP (Higher Education Training Pipeline)

- **Year-over-year training growth** was positive, with total trainees increasing across both programs — evidence that Year 2 investment built on Year 1 momentum rather than starting over.
- **Placement rates** remained a key bottleneck: not all trained providers entered school placements, highlighting a structural gap between supply (trained) and deployment (placed).
- **Provider type mix** skewed toward counselors (44%), followed by psychologists (38%) and social workers (31%), reflecting both grantee capacity and local demand.
- Grantees with **mentorship programs for providers of color** showed higher average training volumes, suggesting equity-focused strategies may support program scale.

### SBMH (K–12 Direct Hiring)

- **Students reached grew substantially** from Year 1 to Year 2, driven both by new hires and by retained providers serving growing caseloads.
- **Student-to-psychologist ratios** remained far above the NASP-recommended 500:1 benchmark for the majority of grantees, underscoring the persistent scale of the need.
- **SEA-funded grantees** (state-level agencies) reached substantially more students per grantee than LEA-funded grantees, reflecting broader geographic reach.
- **Retention is strongly associated with student reach**: grantees that retained more providers in Year 2 reached more students, confirming sustainability of the workforce matters as much as initial hiring.

### Cross-Program

- **Regional variation is substantial**: the West and Southeast showed the largest gains in student reach, while the Central region lagged — suggesting potential for targeted TA investment.
- The combined MHSP + SBMH pipeline demonstrates the logic of a two-sided strategy: universities build the pipeline supply; school agencies deploy it. Year 2 data show the system beginning to function as designed.

---

*Data are synthetic and anonymized. Analyses reflect real methodological approaches applied to simulated federal APR data.*